# FakeNewsNet virality: text prefixes and full-propagation comparison

Text-only inputs from the first 1, 3, 5, or 10 recovered tweets.
Fixed cohort: at least 11 recovered tweets, with matched 60/20/20 splits.
The `full` benchmark uses all recovered text and is retrospective.
Virality targets remain total engagement; future engagement is audited separately.
Run from the repository root; configure windows with `FNN_WINDOWS`.


In [ ]:

from pathlib import Path
import os
import json
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, train_test_split
import sys
sys.path.insert(0, str(Path.cwd() / 'tools'))
from imbalanced_evaluation import evaluate_with_calibration
from fakenewsnet_prefix import DEFAULT_WINDOWS, validate_windows, prepare_features, make_splits, split_fingerprint
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from baseline_backend import build_baselines, fit_evaluate, resolve_backend, require_gpu

if resolve_backend() == "gpu":
    require_gpu()

warnings.filterwarnings('ignore')
os.environ.setdefault('WANDB_DISABLED', 'true')
os.environ.setdefault('WANDB_MODE', 'disabled')

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

ROOT = Path.cwd()  # Run notebook with repository root as working directory
DATA_DIR = ROOT / 'FakeNewsNet' / 'data'

required = [
    DATA_DIR / 'ordered_fake_propagation_paths.jsonl',
    DATA_DIR / 'ordered_real_propagation_paths.jsonl',
    DATA_DIR / 'fake_news_net' / 'ordered_fake_propagation_paths_emb.pt',
    DATA_DIR / 'fake_news_net' / 'ordered_real_propagation_paths_emb.pt',
    DATA_DIR / 'fake_news_net' / 'ordered_fake_propagation_paths_emb_mistral.pt',
    DATA_DIR / 'fake_news_net' / 'ordered_real_propagation_paths_emb_mistral.pt',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Fichiers manquants\n' + '\n'.join(missing))

print('All required files found.')
print('XGBoost available:', HAS_XGB)


In [ ]:

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def load_prefix_features(mode, paths, windows):
    if mode not in ('bert', 'mistral'):
        raise ValueError('mode must be bert or mistral')
    suffix = '_mistral' if mode == 'mistral' else ''
    embeddings = {
        category: torch.load(
            DATA_DIR / 'fake_news_net' / f'ordered_{category}_propagation_paths_emb{suffix}.pt',
            map_location='cpu',
        )
        for category in ('fake', 'real')
    }
    q = float(os.environ.get('VIRALITY_QUANTILE', '0.5'))
    return prepare_features(paths, embeddings, windows, task='virality', quantile=q)


In [ ]:

class FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class MLP(nn.Module):
    def __init__(self, in_dim, hidden=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


def run_mlp_cv(X, y, splits, epochs=12, batch_size=64, lr=1e-3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    rows = []

    for fold, (fit, cal, va) in enumerate(splits, 1):
        np.random.seed(2000 + fold)
        torch.manual_seed(2000 + fold)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(2000 + fold)
        model = MLP(in_dim=X.shape[1]).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.BCEWithLogitsLoss()

        dl_tr = DataLoader(FeatureDataset(X[fit], y[fit]), batch_size=batch_size, shuffle=True)
        dl_cal = DataLoader(FeatureDataset(X[cal], y[cal]), batch_size=batch_size, shuffle=False)
        dl_va = DataLoader(FeatureDataset(X[va], y[va]), batch_size=batch_size, shuffle=False)

        model.train()
        for _ in range(epochs):
            for xb, yb in dl_tr:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                opt.step()

        model.eval()
        def predict_prob(loader):
            probs = []
            with torch.no_grad():
                for xb, _ in loader:
                    probs.append(torch.sigmoid(model(xb.to(device))).cpu().numpy())
            return np.concatenate(probs)

        m = evaluate_with_calibration(y[va], predict_prob(dl_va),
                                      y[cal], predict_prob(dl_cal))
        m['n_fit'] = len(fit)
        m['n_calibration'] = len(cal)
        m['fold'] = fold
        m['split_fingerprint'] = split_fingerprint(fit, cal, va)
        rows.append(m)

    return pd.DataFrame(rows)


def run_baselines_cv(X, y, splits, label="FakeNewsNet"):
    Xn = X.numpy()

    models = build_baselines(logreg_random_state=42)
    if HAS_XGB:
        models['xgboost'] = xgb.XGBClassifier(
            n_estimators=250, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42
        )

    rows = []
    for name, model in models.items():
        for fold, (fit, cal, va) in enumerate(splits, 1):
            m, _ = fit_evaluate(model, Xn, y, fit, cal, va,
                                backend='cpu' if name == 'xgboost' else resolve_backend(),
                                label=f'{label} {name} fold={fold}')
            m['n_fit'] = len(fit)
            m['n_calibration'] = len(cal)
            m['model'] = name
            m['fold'] = fold
            m['split_fingerprint'] = split_fingerprint(fit, cal, va)
            rows.append(m)

    return pd.DataFrame(rows)


In [ ]:

WINDOWS = validate_windows(os.environ.get('FNN_WINDOWS', DEFAULT_WINDOWS))
paths = {category: read_jsonl(DATA_DIR / f'ordered_{category}_propagation_paths.jsonl')
         for category in ('fake', 'real')}
results = []
reference_y, reference_metadata = None, None
prepared = {}
for mode in ['bert', 'mistral']:
    features, y, metadata = load_prefix_features(mode, paths, WINDOWS)
    if reference_y is None:
        reference_y, reference_metadata = y.copy(), metadata.copy()
    elif not np.array_equal(y, reference_y) or metadata != reference_metadata:
        raise ValueError('Encoder cohorts or targets differ; comparison aborted')
    prepared[mode] = features

# Validate both encoder inputs before starting any model training.
y = reference_y
splits = make_splits(y)
for mode, features in prepared.items():
    metadata = reference_metadata
    print(mode, metadata)
    for window in WINDOWS:
        X = torch.from_numpy(features[window])
        print(f'Running {mode}, window={window}, shape={tuple(X.shape)}')
        mlp_df = run_mlp_cv(X, y, splits)
        mlp_df['model'] = f'mlp_{mode}'
        base_df = run_baselines_cv(X, y, splits, label=f"FakeNewsNet {mode} window={window}")
        base_df['model'] = base_df['model'] + f'_{mode}'
        for frame in (mlp_df, base_df):
            frame['observation_window'] = 'full' if window == 'full' else f'k{window}'
            for key, value in metadata.items():
                frame[key] = value
            results.append(frame)

all_results = pd.concat(results, ignore_index=True)
METRICS = ['accuracy', 'f1', 'precision', 'recall', 'auc', 'pr_auc', 'pr_baseline',
           'majority_accuracy', 'majority_f1', 'f1_tuned', 'precision_tuned',
           'recall_tuned', 'accuracy_tuned', 'threshold_tuned']
summary = all_results.groupby(['model', 'observation_window'])[METRICS].mean()
for key, value in reference_metadata.items():
    summary[key] = value
summary = summary.sort_values('f1_tuned', ascending=False)
summary


In [ ]:

Q_LABEL = str(int(round(float(os.environ.get('VIRALITY_QUANTILE', '0.5')) * 100)))
OUT_DIR = ROOT / 'colab_no_embeddings' / 'outputs' / f'fakenewsnet_virality_prefix_q{Q_LABEL}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
all_results.to_csv(OUT_DIR / 'fakenewsnet_virality_merged_cv_results.csv', index=False)
summary.to_csv(OUT_DIR / 'fakenewsnet_virality_merged_summary.csv')
print('Saved results to', OUT_DIR)


In [ ]:
PUB_SUMMARY_FILENAME = 'fakenewsnet_virality_publication_summary.md'

In [ ]:
# Prefix comparisons: no pooling across observation windows.
summary_for_pub = summary.reset_index().round(4)
md_lines = [
    '# FakeNewsNet text prefix evaluation', '',
    'Fixed cohort: at least 11 recovered tweets. Each k uses only its first k texts.',
    'full is a retrospective full-propagation benchmark, not an early predictor.',
    'The cohort is selected using final recovered sequence length; results do not describe all stories.',
    'Cached embedding identity/order must be confirmed with the data owner.',
    '', summary_for_pub.to_markdown(index=False),
]
pub_md_path = OUT_DIR / PUB_SUMMARY_FILENAME
pub_md_path.write_text('\n'.join(md_lines), encoding='utf-8')
print('Saved publication summary to', pub_md_path)
